In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# CẤU HÌNH HỆ THỐNG VÀ DANH SÁCH TẬP DỮ LIỆU CHẠY 3 NGÂN HÀNG
# =====================================================================
datasets = [
    {"ticker": "JPM", "filename": "JPM_master_trees_updated.csv"},
    {"ticker": "WFC", "filename": "WFC_master_trees_updated.csv"},
    {"ticker": "BAC", "filename": "BAC_master_trees_updated.csv"},
]

exclude_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Adj_Close', 'Volume', 'target_return', 'target_direction']
target_reg = 'target_return'
test_years = [2023, 2024, 2025]

# Thùng chứa kết quả Elastic Net toàn cục (nếu bạn cần dùng cho bước so sánh cuối cùng)
elastic_net_master_results = {}

# =====================================================================
# VÒNG LẶP CHÍNH DUYỆT QUA TỪNG NGÂN HÀNG
# =====================================================================
for dataset in datasets:
    ticker = dataset["ticker"]
    filename = dataset["filename"]

    print("\n" + "="*90)
    print(f" KHỞI CHẠY PIPELINE ELASTIC NET CHO NGÂN HÀNG: {ticker} (File: {filename})")
    print("="*90)

    try:
        # 1. Đọc và chuẩn hóa cấu trúc dữ liệu chuỗi thời gian
        df_en = pd.read_csv(filename)
        df_en['Date'] = pd.to_datetime(df_en['Date'])
        df_en = df_en.sort_values('Date').reset_index(drop=True)
    except FileNotFoundError:
        print(f"Không tìm thấy file {filename}. Bỏ qua ngân hàng {ticker}.")
        continue

    # 2. Phân tách danh sách Features và Targets độc lập theo từng file
    features = [c for c in df_en.columns if c not in exclude_cols]
    years = df_en['Date'].dt.year

    all_predictions = []
    all_actuals = []

    print(f"Số lượng Features ghi nhận: {len(features)}")

    # 3. VÒNG LẶP WALK-FORWARD VALIDATION TỪNG NĂM
    for test_yr in test_years:
        print(f"  ▶ Đang xử lý Fold kiểm thử cho năm: {test_yr}")

        # Cấu hình Expanding Window
        train_mask = (years >= 2020) & (years < test_yr)
        test_mask = (years == test_yr)

        X_train = df_en.loc[train_mask, features].values
        y_train = df_en.loc[train_mask, target_reg].values

        X_test = df_en.loc[test_mask, features].values
        y_test = df_en.loc[test_mask, target_reg].values

        # Chuẩn hóa dữ liệu (Feature Scaling) - BẮT BUỘC đối với Elastic Net
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Khởi tạo và huấn luyện mô hình Elastic Net
        model = ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=42, max_iter=2000)
        model_fit = model.fit(X_train_scaled, y_train)

        # Dự báo Out-of-sample cho năm Test hiện tại
        predictions = model_fit.predict(X_test_scaled)

        all_predictions.extend(predictions)
        all_actuals.extend(y_test)

        # Đánh giá nhanh hiệu suất từng năm (Sử dụng ngưỡng động để tính Directional Acc chính xác hơn)
        fold_rmse = np.sqrt(mean_squared_error(y_test, predictions))
        fold_mae = mean_absolute_error(y_test, predictions)

        fold_threshold = np.mean(predictions)
        dir_actual = np.where(y_test > 0, 1, 0)
        dir_pred = np.where(predictions > fold_threshold, 1, 0)
        fold_acc = np.mean(dir_actual == dir_pred) * 100

        # Đếm số lượng feature được giữ lại (Hệ số gán alpha khác 0 nhờ cơ chế trừng phạt L1)
        active_features = np.sum(model_fit.coef_ != 0)

        print(f"    ➔ Kết quả {test_yr}: RMSE = {fold_rmse:.5f} | MAE = {fold_mae:.5f} | Directional Accuracy = {fold_acc:.2f}%")
        print(f"      ↳ Số lượng Features giữ lại thực tế: {active_features}/{len(features)}")

    # Lưu trữ kết quả dự báo 3 năm gộp của Ticker hiện tại vào Master Dict phục vụ so sánh sau này
    elastic_net_master_results[ticker] = {
        "preds": all_predictions,
        "actuals": all_actuals
    }

    # 4. BÁO CÁO HIỆU SUẤT TỔNG THỂ GIAI ĐOẠN OUT-OF-SAMPLE (2023 - 2025) CHO TỪNG TICKER
    overall_rmse = np.sqrt(mean_squared_error(all_actuals, all_predictions))
    overall_mae = mean_absolute_error(all_actuals, all_predictions)

    overall_dir_actual = np.where(np.array(all_actuals) > 0, 1, 0)

    # [CẢI TIẾN QUAN TRỌNG] Sử dụng Mean Threshold tổng thể để triệt tiêu lỗi bias một chiều của mô hình tuyến tính
    overall_threshold = np.mean(all_predictions)
    overall_dir_pred = np.where(np.array(all_predictions) > overall_threshold, 1, 0)
    overall_acc = np.mean(overall_dir_actual == overall_dir_pred) * 100

    print("\n" + "-"*60)
    print(f"BÁO CÁO HIỆU SUẤT TỔNG THỂ ELASTIC NET - {ticker} (2023-2025)")
    print(f" • Overall RMSE: {overall_rmse:.5f}")
    print(f" • Overall MAE : {overall_mae:.5f}")
    print(f" • Overall Directional Accuracy: {overall_acc:.2f}%")
    print("-"*60 + "\n")

print("="*90)
print("PIPELINE HUẤN LUYỆN TỰ ĐỘNG ELASTIC NET CHO CẢ 3 NGÂN HÀNG ĐÃ HOÀN TẤT!")
print("="*90)


 KHỞI CHẠY PIPELINE ELASTIC NET CHO NGÂN HÀNG: JPM (File: JPM_master_trees_updated.csv)
Số lượng Features ghi nhận: 45
  ▶ Đang xử lý Fold kiểm thử cho năm: 2023
    ➔ Kết quả 2023: RMSE = 0.01319 | MAE = 0.00953 | Directional Accuracy = 48.00%
      ↳ Số lượng Features giữ lại thực tế: 19/45
  ▶ Đang xử lý Fold kiểm thử cho năm: 2024
    ➔ Kết quả 2024: RMSE = 0.01506 | MAE = 0.01006 | Directional Accuracy = 44.05%
      ↳ Số lượng Features giữ lại thực tế: 18/45
  ▶ Đang xử lý Fold kiểm thử cho năm: 2025
    ➔ Kết quả 2025: RMSE = 0.01584 | MAE = 0.01096 | Directional Accuracy = 51.41%
      ↳ Số lượng Features giữ lại thực tế: 17/45

------------------------------------------------------------
BÁO CÁO HIỆU SUẤT TỔNG THỂ ELASTIC NET - JPM (2023-2025)
 • Overall RMSE: 0.01474
 • Overall MAE : 0.01018
 • Overall Directional Accuracy: 49.53%
------------------------------------------------------------


 KHỞI CHẠY PIPELINE ELASTIC NET CHO NGÂN HÀNG: WFC (File: WFC_master_trees_updated.